### 1. % 후처리

In [ ]:
import json
import os
from pathlib import Path
import re

# 입력 및 출력 경로 설정
input_path = "../dataset/처리/FRN/ocr_json"
output_path = "../dataset/처리/FRN/ocr_json_preprocessing"

# 출력 디렉토리 생성
os.makedirs(output_path, exist_ok=True)

def process_gasangumri(value):
    """
    가산금리 값의 마지막 자리가 0 또는 09로 끝나는 경우 제거
    """
    if not isinstance(value, str):
        return value
    
    if value.endswith('9') and len(value) > 1:
        return value[:-1]

    if value.endswith('09'):
        return value[:-2]
    
    if value.endswith('96'):
        return value[:-2]
    
    if value.endswith('004'):
        return value[:-3]
    
    return value

def process_json_file(input_file, output_file):
    """
    JSON 파일을 처리하여 가산금리 값을 수정
    """
    try:
        with open(input_file, 'r', encoding='utf-8') as f:
            data = json.load(f)
        
        # '가산금리' 키가 있는 경우 처리
        if '가산금리' in data:
            original_value = data['가산금리']
            processed_value = process_gasangumri(original_value)
            data['가산금리'] = processed_value
            
            if original_value != processed_value:
                print(f"파일: {input_file.name}")
                print(f"  원본 가산금리: {original_value}")
                print(f"  처리된 가산금리: {processed_value}")
        
        # 처리된 데이터를 출력 파일에 저장
        with open(output_file, 'w', encoding='utf-8') as f:
            json.dump(data, f, ensure_ascii=False, indent=2)
            
        return True
        
    except Exception as e:
        print(f"파일 처리 중 오류 발생 {input_file}: {e}")
        return False

# 메인 처리 로직
input_dir = Path(input_path)
output_dir = Path(output_path)

if not input_dir.exists():
    print(f"입력 디렉토리가 존재하지 않습니다: {input_path}")
else:
    json_files = list(input_dir.glob("*.json"))
    
    if not json_files:
        print(f"JSON 파일이 없습니다: {input_path}")
    else:
        processed_count = 0
        total_count = len(json_files)
        
        print(f"총 {total_count}개의 JSON 파일을 처리합니다...")
        
        for json_file in json_files:
            output_file = output_dir / json_file.name
            
            if process_json_file(json_file, output_file):
                processed_count += 1
        
        print(f"\n처리 완료: {processed_count}/{total_count}개 파일 성공")
        print(f"출력 경로: {output_path}")



### 2. 만기정산금액/선도금액 후처리

In [ ]:
import json
import os
from pathlib import Path
import re

# 입력 및 출력 경로 설정
input_path = "../dataset/처리/채권선도/ocr_json"
output_path = "../dataset/처리/채권선도/ocr_json_preprocessing"

# 출력 디렉토리 생성
os.makedirs(output_path, exist_ok=True)

def process_settlement_amount(value):
    """
    만기정산금액 처리 함수
    만기정산금액이 3000000000~30000000000 범위가 되도록 처리
    - 3000000000 이하인 경우 *10 (반복)
    - 30000000000 이상인 경우 /10 (반복)
    - 10~11자리에 될때까지 반복 처리
    """
    if not value or value == "":
        return value
    
    try:
        # 문자열에서 숫자만 추출하고 숫자로 변환
        numeric_str = re.sub(r'[^0-9.]', '', str(value))
        if not numeric_str:
            return value
            
        amount = float(numeric_str)
        original_amount = amount
        
        # 3,000,000,000 ~ 30,000,000,000 범위로 조정
        while amount < 3000000000:
            amount *= 10
        
        while amount > 25000000000:
            amount /= 10
        
        # 정수로 변환 (소수점이 있다면)
        if amount == int(amount):
            amount = int(amount)
        
        if original_amount != amount:
            print(f"만기정산금액 조정: {original_amount} → {amount}")
        
        return str(amount)
        
    except (ValueError, TypeError):
        return value

def process_forward_price(value):
    """
    선도가격 처리 함수
    선도가격이 0.3~3.0 범위가 되도록 처리
    - 0.3보다 이하인 경우 *10 (반복)
    - 3.0보다 이상인 경우 /10 (반복)
    - 0.3~3.0 사이에 들어올때까지 반복 처리
    """
    if not value or value == "":
        return value
    
    try:
        # 문자열에서 숫자만 추출하고 숫자로 변환
        numeric_str = re.sub(r'[^0-9.]', '', str(value))
        if not numeric_str:
            return value
            
        price = float(numeric_str)
        original_price = price
        
        # 0.3 ~ 3.0 범위로 조정
        while price < 0.3 and price > 0:
            price *= 10
        
        while price > 2.5:
            price /= 10
        
        # 소수점 처리
        price = round(price, 6)
        
        if original_price != price:
            print(f"선도가격 조정: {original_price} → {price}")
        
        return str(price)
        
    except (ValueError, TypeError):
        return value

def process_json_file(input_file, output_file):
    """
    JSON 파일을 처리하여 만기정산금액과 선도가격을 수정
    """
    try:
        with open(input_file, 'r', encoding='utf-8') as f:
            data = json.load(f)
        
        changes_made = False
        
        # '만기정산금액' 키가 있는 경우 처리
        if '만기정산금액' in data:
            original_value = data['만기정산금액']
            processed_value = process_settlement_amount(original_value)
            if original_value != processed_value:
                data['만기정산금액'] = processed_value
                changes_made = True
        
        # '선도가격' 키가 있는 경우 처리
        if '선도가격' in data:
            original_value = data['선도가격']
            processed_value = process_forward_price(original_value)
            if original_value != processed_value:
                data['선도가격'] = processed_value
                changes_made = True
        
        if changes_made:
            print(f"파일 처리: {input_file.name}")
        
        # 처리된 데이터를 출력 파일에 저장
        with open(output_file, 'w', encoding='utf-8') as f:
            json.dump(data, f, ensure_ascii=False, indent=2)
            
        return True
        
    except Exception as e:
        print(f"파일 처리 중 오류 발생 {input_file}: {e}")
        return False

# 메인 처리 로직
input_dir = Path(input_path)
output_dir = Path(output_path)

if not input_dir.exists():
    print(f"입력 디렉토리가 존재하지 않습니다: {input_path}")
else:
    json_files = list(input_dir.glob("*.json"))
    
    if not json_files:
        print(f"JSON 파일이 없습니다: {input_path}")
    else:
        processed_count = 0
        total_count = len(json_files)
        
        print(f"총 {total_count}개의 JSON 파일을 처리합니다...")
        print("="*50)
        
        for json_file in json_files:
            output_file = output_dir / json_file.name
            
            if process_json_file(json_file, output_file):
                processed_count += 1
        
        print("="*50)
        print(f"\n처리 완료: {processed_count}/{total_count}개 파일 성공")
        print(f"출력 경로: {output_path}")

# Cell 7: 기초자산 후처리 - 문자 교정 (o/O → 0, t → 1)

import json
import os
from pathlib import Path
import re

def process_underlying_asset(value):
    """
    기초자산 값에서 OCR 오류로 인한 문자 교정
    - 소문자 o, 대문자 O → 0으로 변환
    - 소문자 t → 1로 변환
    """
    if not value or value == "":
        return value
    
    try:
        # 문자열로 변환
        processed_value = str(value)
        original_value = processed_value
        
        # 문자 교정 규칙 적용
        processed_value = processed_value.replace('o', '0')  # 소문자 o → 0
        processed_value = processed_value.replace('O', '0')  # 대문자 O → 0
        processed_value = processed_value.replace('t', '1')  # 소문자 t → 1
        
        if original_value != processed_value:
            print(f"기초자산 교정: '{original_value}' → '{processed_value}'")
        
        return processed_value
        
    except Exception as e:
        print(f"기초자산 처리 중 오류: {e}")
        return value

def process_json_file_for_underlying(input_file, output_file):
    """
    JSON 파일을 처리하여 기초자산 값을 교정
    """
    try:
        with open(input_file, 'r', encoding='utf-8') as f:
            data = json.load(f)
        
        changes_made = False
        
        # '기초자산' 키가 있는 경우 처리
        if '기초자산' in data:
            original_value = data['기초자산']
            processed_value = process_underlying_asset(original_value)
            if original_value != processed_value:
                data['기초자산'] = processed_value
                changes_made = True
        
        if changes_made:
            print(f"파일 처리: {input_file.name}")
        
        # 처리된 데이터를 출력 파일에 저장
        with open(output_file, 'w', encoding='utf-8') as f:
            json.dump(data, f, ensure_ascii=False, indent=2)
            
        return True
        
    except Exception as e:
        print(f"파일 처리 중 오류 발생 {input_file}: {e}")
        return False

def batch_process_underlying_assets(input_dir, output_dir):
    """
    디렉터리 내 모든 JSON 파일의 기초자산을 일괄 처리
    """
    # 입력 및 출력 경로 설정
    input_path = Path(input_dir)
    output_path = Path(output_dir)
    
    # 출력 디렉토리 생성
    os.makedirs(output_path, exist_ok=True)
    
    # 메인 처리 로직
    if not input_path.exists():
        print(f"입력 디렉토리가 존재하지 않습니다: {input_dir}")
        return
    
    json_files = list(input_path.glob("*.json"))
    
    if not json_files:
        print(f"JSON 파일이 없습니다: {input_dir}")
        return
    
    processed_count = 0
    total_count = len(json_files)
    
    print(f"총 {total_count}개의 JSON 파일을 처리합니다...")
    print("="*50)
    
    for json_file in json_files:
        output_file = output_path / json_file.name
        
        if process_json_file_for_underlying(json_file, output_file):
            processed_count += 1
    
    print("="*50)
    print(f"\n처리 완료: {processed_count}/{total_count}개 파일 성공")
    print(f"출력 경로: {output_dir}")

# 실행 예시
if __name__ == "__main__":
    # 채권선도 데이터 처리
    input_directory = "../dataset/처리/채권선도/ocr_json_preprocessing"
    output_directory = "../dataset/처리/채권선도/ocr_json_preprocessing"
    
    print("=== 채권선도 기초자산 후처리 시작 ===")
    batch_process_underlying_assets(input_directory, output_directory)